# 16 — File Formats in PySpark: CSV, JSON, Avro, ORC, Parquet & Delta

**What you will learn:**
- How each format stores data and its trade-offs
- Reading and writing each format with all important options
- Schema inference vs explicit schema for each format
- Compression codecs supported by each format
- Predicate pushdown and column pruning support
- Nested / complex data handling (JSON, Avro, ORC, Parquet)
- Delta vs Parquet — what Delta adds on top
- Delta Tables: managed vs external, DDL, DML, time travel, CDF
- Format comparison matrix and when to use which

## Format Overview

| Format | Type | Columnar | Schema Embedded | Compression | ACID | Best For |
|---|---|---|---|---|---|---|
| CSV | Row | No | No | Optional | No | Data exchange, Excel exports |
| JSON | Row | No | No (inferred) | Optional | No | APIs, semi-structured / nested data |
| Avro | Row | No | Yes (in file) | Block-level | No | Kafka, schema evolution, streaming |
| ORC | Columnar | Yes | Yes | Stripe-level | No | Hive workloads, heavy aggregations |
| Parquet | Columnar | Yes | Yes | Row group | No | Analytics, data lake standard |
| Delta | Columnar + Log | Yes | Yes | Row group | Yes | Lakehouse, ACID, time travel, CDC |

## Setup — Sample Data

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DoubleType, BooleanType, ArrayType
)

schema = StructType([
    StructField("emp_id",     IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("hire_date",  StringType(),  True),
    StructField("is_active",  BooleanType(), True),
    StructField("skills",     ArrayType(StringType()), True),
])

data = [
    (1, "Alice",   "Engineering", 95000.0,  "2022-01-15", True,  ["Python","Spark","SQL"]),
    (2, "Bob",     "Marketing",   72000.0,  "2021-06-01", True,  ["Excel","PowerBI"]),
    (3, "Charlie", "Engineering", 105000.0, "2020-03-20", True,  ["Java","Spark","Scala"]),
    (4, "Diana",   "HR",          68000.0,  "2023-09-10", False, ["Excel","SAP"]),
    (5, "Eve",     "Marketing",   78000.0,  "2022-11-05", True,  ["Python","Tableau"]),
    (6, "Frank",   "Engineering", 88000.0,  "2021-04-12", True,  ["Python","Spark","AWS"]),
]

df = spark.createDataFrame(data, schema=schema)
df.show(truncate=False)
df.printSchema()

## 1. CSV — Comma Separated Values

**Characteristics:**
- Plain text, row-oriented, human-readable
- No schema embedded — types must be inferred or provided explicitly
- No column skipping, no predicate pushdown
- Arrays/Maps cannot be natively written — flatten complex columns first
- Best for: data exchange with non-Spark systems, Excel/BI tool exports

In [ ]:
# CSV does not support ArrayType — flatten before writing
df_flat = df.drop("skills")

# ── WRITE CSV ──
(df_flat.write
    .mode("overwrite")
    .option("header",    "true")       # write column names as first row
    .option("delimiter", ",")          # default comma; use "\t" for TSV
    .option("quote",     '"')          # quoting char for fields with commas
    .option("nullValue", "NULL")       # write nulls as the string "NULL"
    .option("dateFormat","yyyy-MM-dd")
    .csv("/tmp/formats/csv/")
)
display(dbutils.fs.ls("/tmp/formats/csv/"))  # multiple part-* files

In [ ]:
# ── READ CSV — infer schema (slow; types may be wrong) ──
df_csv_infer = (spark.read
    .option("header",      "true")
    .option("inferSchema", "true")
    .option("nullValue",   "NULL")
    .csv("/tmp/formats/csv/")
)
df_csv_infer.printSchema()
df_csv_infer.show()

In [ ]:
# ── READ CSV — explicit schema (ALWAYS prefer in production) ──
csv_schema = StructType([
    StructField("emp_id",     IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("hire_date",  StringType(),  True),
    StructField("is_active",  BooleanType(), True),
])

df_csv = (spark.read
    .option("header",                    "true")
    .option("nullValue",                 "NULL")
    .option("multiLine",                 "false")
    .option("ignoreLeadingWhiteSpace",   "true")
    .option("ignoreTrailingWhiteSpace",  "true")
    .option("mode",                      "DROPMALFORMED")  # PERMISSIVE | DROPMALFORMED | FAILFAST
    .schema(csv_schema)
    .csv("/tmp/formats/csv/")
)
df_csv.printSchema()
df_csv.show()

In [ ]:
# ── COMPRESSED CSV ──
(df_flat.write
    .mode("overwrite")
    .option("header",      "true")
    .option("compression", "gzip")     # none | gzip | bz2 | deflate | lz4 | snappy
    .csv("/tmp/formats/csv_gz/")
)
# Spark auto-detects .gz extension on read
df_gz = spark.read.option("header","true").option("inferSchema","true")              .csv("/tmp/formats/csv_gz/")
df_gz.show()

# ── ADD SOURCE FILE NAME FOR AUDIT ──
from pyspark.sql.functions import input_file_name
df_csv.withColumn("source_file", input_file_name()).show(3, truncate=False)

## 2. JSON — JavaScript Object Notation

**Characteristics:**
- Row-oriented, schema inferred from values
- **JSON Lines** (one JSON object per line) is the Spark default
- Native support for nested structures (struct, array, map)
- Larger than binary formats; no column skipping
- Best for: REST API responses, event logs, semi-structured / nested data

In [ ]:
# ── WRITE JSON (JSON Lines — one object per line) ──
(df.write
    .mode("overwrite")
    .option("compression", "none")    # none | gzip | bz2 | deflate | lz4 | snappy
    .json("/tmp/formats/json/")
)
print("JSON written.")

In [ ]:
# ── READ JSON — inferred schema ──
df_json = spark.read.json("/tmp/formats/json/")
df_json.printSchema()
df_json.show(truncate=False)

# ── READ JSON — explicit schema ──
df_json_typed = spark.read.schema(schema).json("/tmp/formats/json/")
df_json_typed.printSchema()

In [ ]:
# ── NESTED JSON ──
nested_data = [
    '{"order_id":1001,"customer":{"id":1,"name":"Alice","tier":"Gold"},"items":[{"sku":"A1","qty":2},{"sku":"B2","qty":1}],"total":299.99}',
    '{"order_id":1002,"customer":{"id":2,"name":"Bob","tier":"Silver"},"items":[{"sku":"C3","qty":5}],"total":149.50}',
]
dbutils.fs.put("/tmp/formats/nested.json", "\n".join(nested_data), overwrite=True)

df_nested = spark.read.json("/tmp/formats/nested.json")
df_nested.printSchema()
df_nested.show(truncate=False)

# Access nested fields using dot notation and array indexing
df_nested.select(
    F.col("order_id"),
    F.col("customer.name").alias("customer_name"),
    F.col("customer.tier"),
    F.col("total"),
    F.col("items")[0]["sku"].alias("first_item_sku"),
    F.explode(F.col("items")).alias("item")   # one row per item
).select("order_id","customer_name","item.sku","item.qty","total").show()

In [ ]:
# ── MULTI-LINE JSON (entire file is one JSON array) ──
multi = '[{"emp_id":1,"name":"Alice"},{"emp_id":2,"name":"Bob"}]'
dbutils.fs.put("/tmp/formats/multi.json", multi, overwrite=True)

df_multi = spark.read.option("multiLine","true").json("/tmp/formats/multi.json")
df_multi.show()

# Corrupt record handling
# .option("mode","PERMISSIVE")    — default; bad row → _corrupt_record column
# .option("mode","DROPMALFORMED") — silently drop bad rows
# .option("mode","FAILFAST")      — throw on first bad row

## 3. Avro

**Characteristics:**
- Row-oriented binary format with schema (JSON) embedded in the file header
- Excellent **schema evolution** — add/remove fields with defaults, readers stay compatible
- Used heavily with **Apache Kafka** (each message is Avro-encoded)
- Block-level compression; good for streaming / sequential reads
- Best for: Kafka pipelines, schema evolution use cases, row-level streaming

In [ ]:
# Avro requires spark-avro (included in Databricks by default)

# ── WRITE AVRO ──
(df.write
    .format("avro")
    .mode("overwrite")
    .option("compression", "snappy")   # none | deflate | snappy | bzip2 | xz
    .save("/tmp/formats/avro/")
)
print("Avro written.")

In [ ]:
# ── READ AVRO ──
df_avro = spark.read.format("avro").load("/tmp/formats/avro/")
df_avro.printSchema()
df_avro.show(truncate=False)

In [ ]:
# ── AVRO WITH EXPLICIT AVRO SCHEMA (project specific fields) ──
avro_schema_str = '''
{
  "type": "record",
  "name": "Employee",
  "fields": [
    {"name": "emp_id",     "type": ["null","int"],    "default": null},
    {"name": "name",       "type": ["null","string"], "default": null},
    {"name": "department", "type": ["null","string"], "default": null},
    {"name": "salary",     "type": ["null","double"], "default": null}
  ]
}
'''

# Read only the 4 fields defined in the Avro schema — column projection at source
df_projected = (spark.read
    .format("avro")
    .option("avroSchema", avro_schema_str)
    .load("/tmp/formats/avro/")
)
df_projected.printSchema()
df_projected.show()

In [ ]:
# ── AVRO SCHEMA EVOLUTION ──
# Avro allows adding new fields with defaults — old files remain readable

df_v2 = df.withColumn("location", F.lit("NYC"))   # new column added
(df_v2.write.format("avro").mode("overwrite").save("/tmp/formats/avro_v2/"))

df_read_v2 = spark.read.format("avro").load("/tmp/formats/avro_v2/")
df_read_v2.printSchema()
df_read_v2.show(truncate=False)

# Key advantage: Avro readers can read older files missing the new field
# by supplying a default value in the reader schema — no migration needed

## 4. ORC — Optimized Row Columnar

**Characteristics:**
- Columnar binary format designed for **Apache Hive**
- Stripes (row groups) with built-in lightweight indexes (min/max, bloom filters)
- Strong compression with **Zlib** (default) or **Snappy**
- Predicate pushdown and column pruning fully supported
- Best for: Hive / EMR workloads, heavy analytical aggregations

In [ ]:
# ── WRITE ORC ──
(df.write
    .format("orc")
    .mode("overwrite")
    .option("compression", "snappy")   # none | zlib (default) | snappy | lzo | lz4 | zstd
    .save("/tmp/formats/orc/")
)
print("ORC written.")

In [ ]:
# ── READ ORC ──
df_orc = spark.read.format("orc").load("/tmp/formats/orc/")
df_orc.printSchema()
df_orc.show(truncate=False)

In [ ]:
# ── ORC PREDICATE PUSHDOWN ──
# ORC stores min/max per stripe — Spark skips non-matching stripes

df_orc_filter = (spark.read.format("orc").load("/tmp/formats/orc/")
                 .filter(F.col("salary") > 90000))
df_orc_filter.explain(mode="formatted")  # look for PushedFilters
df_orc_filter.show()

In [ ]:
# ── ORC vs PARQUET SIZE COMPARISON ──
import time

df_large = spark.createDataFrame(
    [(i, f"emp_{i}", ["Engineering","Marketing","HR"][i%3],
      50000.0 + (i * 137 % 70000)) for i in range(1, 300001)],
    ["id","name","department","salary"]
)

t0 = time.time()
df_large.write.format("orc").mode("overwrite").save("/tmp/formats/orc_large/")
orc_t = time.time()-t0

t0 = time.time()
df_large.write.format("parquet").mode("overwrite").save("/tmp/formats/parquet_large/")
pq_t = time.time()-t0

orc_sz     = sum(f.size for f in dbutils.fs.ls("/tmp/formats/orc_large/")     if ".orc"     in f.name)
parquet_sz = sum(f.size for f in dbutils.fs.ls("/tmp/formats/parquet_large/") if ".parquet" in f.name)

print(f"ORC     — write: {orc_t:.2f}s  | size: {orc_sz/1024:.0f} KB")
print(f"Parquet — write: {pq_t:.2f}s   | size: {parquet_sz/1024:.0f} KB")

## 5. Parquet

**Characteristics:**
- Columnar binary format — the **de-facto standard** for data lakes
- Schema embedded in file footer; row group min/max statistics
- Column encoding + dictionary encoding + RLE → excellent compression
- Predicate pushdown at row-group level; column pruning reads only needed columns
- Supported by virtually every big data tool (Spark, Hive, Presto, Athena, BigQuery, Snowflake)
- Best for: data lake storage, analytics, cross-tool interoperability

In [ ]:
# ── WRITE PARQUET ──
(df.write
    .format("parquet")          # or shorthand: .parquet(path)
    .mode("overwrite")
    .option("compression", "snappy")   # none | snappy (default) | gzip | lzo | brotli | lz4 | zstd
    .save("/tmp/formats/parquet/")
)
print("Parquet written.")

In [ ]:
# ── READ PARQUET ──
df_pq = spark.read.parquet("/tmp/formats/parquet/")
df_pq.printSchema()
df_pq.show(truncate=False)

In [ ]:
# ── COLUMN PRUNING — only selected columns read from disk ──
spark.read.parquet("/tmp/formats/parquet/")     .select("emp_id","salary").explain(mode="formatted")
# ReadSchema: struct<emp_id:int, salary:double>  ← only 2 cols, not all 7

# ── PREDICATE PUSHDOWN — filters applied inside Parquet reader ──
spark.read.parquet("/tmp/formats/parquet/")     .filter(F.col("salary") > 90000).explain(mode="formatted")
# PushedFilters: [IsNotNull(salary), GreaterThan(salary,90000.0)]

In [ ]:
# ── PARTITIONED PARQUET ──
(df.write
    .format("parquet")
    .mode("overwrite")
    .partitionBy("department")
    .option("compression","snappy")
    .save("/tmp/formats/parquet_partitioned/")
)
display(dbutils.fs.ls("/tmp/formats/parquet_partitioned/"))

# Partition pruning — only Engineering folder scanned
spark.read.parquet("/tmp/formats/parquet_partitioned/")     .filter(F.col("department") == "Engineering").show()

In [ ]:
# ── SCHEMA MERGE — read files with different schemas ──
df.drop("skills").write.mode("overwrite").parquet("/tmp/formats/pq_merge/v1/")
df.drop("is_active").write.mode("overwrite").parquet("/tmp/formats/pq_merge/v2/")

df_merged = (spark.read
    .option("mergeSchema","true")     # union of schemas; missing cols = null
    .parquet("/tmp/formats/pq_merge/")
)
df_merged.printSchema()
df_merged.show(truncate=False)

## 6. Delta Format

**Characteristics:**
- Built on top of **Parquet files** + a **transaction log** (`_delta_log/`)
- Adds: ACID transactions, DML (UPDATE/DELETE/MERGE), time travel, schema enforcement
- Transaction log records every commit — enables auditing, rollback, and CDF
- File-level min/max statistics for data skipping beyond row-group level
- Best for: Lakehouse architecture — any table that requires mutations or CDC

In [ ]:
# ── WRITE DELTA ──
(df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save("/tmp/formats/delta/")
)

# The _delta_log folder is what makes it Delta
display(dbutils.fs.ls("/tmp/formats/delta/_delta_log/"))

In [ ]:
# ── READ DELTA ──
df_delta = spark.read.format("delta").load("/tmp/formats/delta/")
df_delta.printSchema()
df_delta.show(truncate=False)

In [ ]:
# ── WHAT IS INSIDE THE TRANSACTION LOG ──
# Each commit is a JSON file that records adds/removes/commitInfo
spark.read.json("/tmp/formats/delta/_delta_log/00000000000000000000.json")     .select("commitInfo.operation","add.path","add.size","add.stats")     .show(truncate=False)

In [ ]:
# ── DELTA APPEND ──
new_rows = spark.createDataFrame(
    [(7,"Grace","HR",71000.0,"2022-07-30",True,["HR","Workday"])],
    schema=schema
)
new_rows.write.format("delta").mode("append").save("/tmp/formats/delta/")
print("Row count after append:", spark.read.format("delta").load("/tmp/formats/delta/").count())

In [ ]:
# ── UPDATE / DELETE / MERGE ──
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, "/tmp/formats/delta/")

# UPDATE
dt.update(condition=F.col("department") == "Engineering",
          set={"salary": F.col("salary") * F.lit(1.10)})

# DELETE
dt.delete(condition=F.col("is_active") == False)

# MERGE (upsert — most important Delta operation)
updates = spark.createDataFrame(
    [(2,"Bob","Digital Marketing",76000.0,"2021-06-01",True,["Excel","PowerBI","Tableau"]),
     (8,"Hank","Engineering",92000.0,"2024-01-01",True,["Python","Go"])],
    schema=schema
)
(dt.alias("t")
   .merge(updates.alias("s"), "t.emp_id = s.emp_id")
   .whenMatchedUpdateAll()
   .whenNotMatchedInsertAll()
   .execute())

spark.read.format("delta").load("/tmp/formats/delta/").show(truncate=False)

In [ ]:
# ── TIME TRAVEL ──
spark.sql("DESCRIBE HISTORY delta.`/tmp/formats/delta/`").show(truncate=False)

# Read version 0 — the original write
df_v0 = spark.read.format("delta").option("versionAsOf",0).load("/tmp/formats/delta/")
print("Version 0 rows:", df_v0.count())
df_v0.show()

In [ ]:
# ── SCHEMA EVOLUTION ──
# Default: Delta rejects writes with new columns (schema enforcement)
try:
    bad = spark.createDataFrame([(99,"Test","Ops",80000.0,"2024-01-01",True,[],"extra_col")],
                                 ["emp_id","name","department","salary","hire_date","is_active","skills","education"])
    bad.write.format("delta").mode("append").save("/tmp/formats/delta/")
except Exception as e:
    print("Schema enforcement:", str(e)[:100])

# Allow schema evolution with mergeSchema
evolved = spark.createDataFrame(
    [(10,"Jake","Finance",82000.0,"2024-03-01",True,["Finance","Excel"],"MBA")],
    ["emp_id","name","department","salary","hire_date","is_active","skills","education"]
)
evolved.write.format("delta").mode("append").option("mergeSchema","true")     .save("/tmp/formats/delta/")
spark.read.format("delta").load("/tmp/formats/delta/").show(truncate=False)

## 7. Delta Tables — Managed & External

Registering Delta files as a **table** in the Metastore gives them a SQL name.

| | Managed Table | External Table |
|---|---|---|
| Storage | Controlled by Databricks | You specify the ADLS path |
| `DROP TABLE` | Deletes metadata AND data files | Deletes metadata only; data stays |
| Use when | You want Databricks to manage storage | Data already lives at a known path |

In [ ]:
# ── MANAGED DELTA TABLE ──
df.write.format("delta").mode("overwrite").saveAsTable("default.emp_managed")
spark.sql("DESCRIBE DETAIL default.emp_managed").select("name","location","format").show(truncate=False)

In [ ]:
# ── EXTERNAL DELTA TABLE ──
(df.write
    .format("delta")
    .mode("overwrite")
    .option("path", "/tmp/formats/delta_external/")
    .saveAsTable("default.emp_external"))
spark.sql("DESCRIBE DETAIL default.emp_external").select("name","location","format").show(truncate=False)

In [ ]:
# ── CREATE TABLE WITH DDL ──
spark.sql('''
    CREATE TABLE IF NOT EXISTS default.emp_ddl (
        emp_id     INT      NOT NULL,
        name       STRING,
        department STRING,
        salary     DOUBLE,
        hire_date  STRING,
        is_active  BOOLEAN
    )
    USING DELTA
    LOCATION "/tmp/formats/delta_ddl/"
    COMMENT "Employees table created via DDL"
    TBLPROPERTIES (
        "delta.enableChangeDataFeed"       = "true",
        "delta.autoOptimize.optimizeWrite" = "true",
        "delta.autoOptimize.autoCompact"   = "true"
    )
''')
spark.sql("SHOW TABLES IN default").show()

In [ ]:
# ── SQL DML on Delta Tables ──
spark.sql('''
    INSERT INTO default.emp_managed VALUES
    (8, "Hank", "Engineering", 92000.0, "2023-01-01", true,  array("Python","Spark")),
    (9, "Ivy",  "Marketing",   74000.0, "2022-05-15", true,  array("Excel"))
''')

spark.sql('''
    UPDATE default.emp_managed
    SET salary = salary * 1.12
    WHERE department = "Engineering"
''')

spark.sql("DELETE FROM default.emp_managed WHERE is_active = false")

spark.sql("SELECT * FROM default.emp_managed ORDER BY emp_id").show(truncate=False)

In [ ]:
# ── OPTIMIZE + VACUUM ──
spark.sql("OPTIMIZE default.emp_managed ZORDER BY (department, emp_id)")
spark.sql("VACUUM default.emp_managed RETAIN 168 HOURS")   # 7-day retention

# ── TABLE HISTORY ──
spark.sql("DESCRIBE HISTORY default.emp_managed").show(truncate=False)

In [ ]:
# ── CHANGE DATA FEED (CDF) ──
spark.sql('ALTER TABLE default.emp_managed SET TBLPROPERTIES ("delta.enableChangeDataFeed" = "true")')
spark.sql('UPDATE default.emp_managed SET salary = 100000 WHERE emp_id = 1')

# Read only the changed rows — great for incremental downstream pipelines
cdf = (spark.read.format("delta")
       .option("readChangeFeed",  "true")
       .option("startingVersion", 0)
       .table("default.emp_managed"))
cdf.select("emp_id","name","salary","_change_type","_commit_version").show()

## 8. Format Comparison Matrix

| Feature | CSV | JSON | Avro | ORC | Parquet | Delta |
|---|---|---|---|---|---|---|
| **Storage type** | Row | Row | Row | Columnar | Columnar | Columnar |
| **Schema in file** | No | No | Yes | Yes | Yes | Yes + log |
| **Compression** | Optional | Optional | Block | Stripe | Row group | Row group |
| **Column pruning** | No | No | No | Yes | Yes | Yes |
| **Predicate pushdown** | No | No | No | Yes | Yes | Yes + stats |
| **Complex types** | No | Yes | Yes | Yes | Yes | Yes |
| **ACID / DML** | No | No | No | No | No | Yes |
| **Time travel** | No | No | No | No | No | Yes |
| **Schema evolution** | Manual | Manual | Native | Limited | Via option | Native |
| **Typical size** | 1× | 1.2× | 0.7× | 0.4× | 0.4× | ~0.4× |

## 9. When to Use Which Format

| Scenario | Recommended Format |
|---|---|
| Exchange data with Excel / BI tools | CSV |
| Ingest from REST APIs | JSON |
| Kafka → Spark streaming pipeline | Avro |
| Hive / EMR analytical workloads | ORC |
| Analytics data lake (read-heavy) | Parquet |
| Lakehouse with updates / deletes | **Delta** |
| Incremental / CDC pipelines | **Delta** (with CDF) |
| ML feature store | Parquet or Delta |
| Long-term archive | Parquet (Snappy or Zstd) |

## Key Takeaways

- **CSV / JSON** — human-readable, no column skipping — use only at ingestion boundaries
- **Avro** — best for Kafka / streaming; schema evolution is first-class
- **ORC** — optimized for Hive; excellent compression; less common outside Hive ecosystems
- **Parquet** — data lake gold standard; columnar, efficient, universally supported
- **Delta** — Parquet + transaction log; adds ACID, DML, time travel; use for all mutable tables
- **Always use explicit schema** at read time — never rely on `inferSchema` in production
- **Always compress** — Snappy for speed, Zstd/Gzip for maximum size reduction
- **Partition large tables** by a low-cardinality column (date, region, dept) for partition pruning